# 02. MostPopular baseline

Первая модель и определения top-K метрик на январском validation.

## 1. Данные и конфиг

In [1]:
from pathlib import Path
import json
import math

import pandas as pd

search = [Path.cwd(), *Path.cwd().parents]
candidates = [p for base in search for p in (base, base / 'movielens-reccomender', base / 'ml-1m' / 'movielens-reccomender')]
ROOT = next(p for p in candidates if (p / 'src' / 'movielens_recsys' / 'data.py').exists())

from movielens_recsys.data import load_movielens

config = json.loads((ROOT / 'configs' / 'experiment.json').read_text(encoding='utf-8'))
ratings, movies, users = load_movielens(ROOT / config['data_dir'])

## 2. Временное разделение
История заканчивается до января; validation - январь, test - февраль.

In [2]:

history = ratings[
    ratings["datetime"] < config['split']['validation_start']
]
validation = ratings[
    (ratings["datetime"] >= config['split']['validation_start'])
    & (ratings["datetime"] < config['split']['test_start'])
]
test = ratings[
    (ratings["datetime"] >= config['split']['test_start'])
    & (ratings["datetime"] < config['split']['test_end'])
]

In [3]:
print(history['datetime'].max())
print(validation['datetime'].min())
print(validation['datetime'].max())
print(test['datetime'].min())

2000-12-31 23:43:21+00:00
2001-01-01 00:07:45+00:00
2001-01-31 22:11:05+00:00
2001-02-01 00:17:32+00:00


## 3. Популярность
Число оценок 4–5 из истории определяет порядок фильмов.

In [4]:
positive_history = history[history["rating"] >= config['positive_threshold']]
positive_history.head()

,user_id,movie_id,rating,timestamp,datetime
0,1,1193,5,978300760,2000-12-31 22:12:40+00:00
3,1,3408,4,978300275,2000-12-31 22:04:35+00:00
6,1,1287,5,978302039,2000-12-31 22:33:59+00:00
7,1,2804,5,978300719,2000-12-31 22:11:59+00:00
8,1,594,4,978302268,2000-12-31 22:37:48+00:00


In [5]:
popularity = positive_history.groupby("movie_id").size()
popularity = popularity.reset_index(name="popularity").sort_values(
        by=["popularity", "movie_id"],
        ascending=[False, True],
    )
popularity.head()

,movie_id,popularity
2499,2858,2749
242,260,2481
1027,1196,2393
1734,2028,2162
1029,1198,2137


## 4. Ранее оценённые фильмы
Seen включает любые оценки, в том числе 1–3.

In [6]:
seen_movies = history.groupby("user_id")["movie_id"].apply(set).to_dict()

In [7]:
def get_popularity_rec(top_n, seen, popular_movies):
    recs = []
    for movie in popular_movies:
        if movie not in seen:
            recs.append(movie)
        if len(recs) == top_n:
            break
    return recs

## 5. Ground truth и пользователи
Правильные ответы - все положительные январские фильмы. Когорта содержит пользователей с историей и непустым ground truth.

In [8]:
ground_truth = validation[validation["rating"] >= config['positive_threshold']].groupby("user_id")["movie_id"].apply(set).to_dict()

In [9]:
eligible_users = (
    set(ground_truth.keys())
    & set(history["user_id"].unique())
)

## 6. Рекомендации MostPopular
Из общего рейтинга популярности исключаются seen-фильмы каждого пользователя.

In [10]:
K = 10
popular_movie_ids = popularity["movie_id"].tolist()

recommendations = {
    user_id: get_popularity_rec(
        top_n=K,
        seen=seen_movies[user_id],
        popular_movies=popular_movie_ids,
    )
    for user_id in eligible_users
}

assert all(len(recs) == K for recs in recommendations.values())
assert all(
    set(recs).isdisjoint(seen_movies[user_id])
    for user_id, recs in recommendations.items()
)

## 7. Оценка качества
Precision, Recall, HitRate и NDCG усредняются по пользователям. CatalogCoverage описывает долю исторического каталога во всей выдаче.

In [11]:
def evaluate_recommendations(
    recommendations,
    ground_truth,
    eligible_users,
    catalog,
    k=10,
):
    rows = []

    for user_id in eligible_users:
        recs = recommendations[user_id][:k]
        relevant = ground_truth[user_id]

        hits = sum(movie_id in relevant for movie_id in recs)
        precision = hits / k
        recall = hits / len(relevant)
        hitrate = float(hits > 0)

        dcg = sum(
            1 / math.log2(rank + 1)
            for rank, movie_id in enumerate(recs, start=1)
            if movie_id in relevant
        )

        ideal_hits = min(len(relevant), k)

        idcg = sum(
            1 / math.log2(rank + 1)
            for rank in range(1, ideal_hits + 1)
        )

        ndcg = dcg / idcg

        rows.append({
            "user_id": user_id,
            "relevant_count": len(relevant),
            "recommendation_count": len(recs),
            "hits": hits,
            f"precision_at_{k}": precision,
            f"recall_at_{k}": recall,
            f"hitrate_at_{k}": hitrate,
            f"ndcg_at_{k}": ndcg,
        })

    metrics_by_user = pd.DataFrame(rows)

    metric_columns = [
        f"precision_at_{k}",
        f"recall_at_{k}",
        f"hitrate_at_{k}",
        f"ndcg_at_{k}",
    ]

    summary = (
        metrics_by_user[metric_columns]
        .mean()
        .to_frame("MostPopular")
    )

    recommended_items = {
        movie_id
        for user_id in eligible_users
        for movie_id in recommendations[user_id][:k]
    }

    catalog_coverage = (
        len(recommended_items)
        / len(catalog)
    )

    summary.loc[f"catalog_coverage_at_{k}"] = catalog_coverage
    summary.loc["eligible_users"] = len(eligible_users)
    summary.loc["catalog_size"] = len(catalog)
    summary.loc["unique_recommended_items"] = len(recommended_items)

    return metrics_by_user, summary

### Интерпретация baseline

Precision@10=0.0913: в среднем около 0.91 попадания в десяти рекомендациях. Recall@10=0.0497: найдены около 5% будущих положительных фильмов. HitRate@10=0.3597: хотя бы одно попадание у 36% пользователей. NDCG@10=0.1088 учитывает также позиции попаданий.

In [12]:
catalog = set(history["movie_id"])
metrics_by_user, summary = evaluate_recommendations(
    recommendations=recommendations,
    ground_truth=ground_truth,
    eligible_users=eligible_users,
    catalog=catalog,
    k=K,
)
summary

,MostPopular
precision_at_10,0.091304
recall_at_10,0.049683
hitrate_at_10,0.359684
ndcg_at_10,0.108800
catalog_coverage_at_10,0.036705
eligible_users,506.000000
catalog_size,3678.000000
unique_recommended_items,135.000000


## MostPopular при разных K

Одинаковая когорта и полный ground truth для K=10/50/100/250. Популярность - число положительных исторических оценок; порядок popularity DESC, movie_id ASC. Таблица сохраняется для сравнения в ноутбуке 03.

In [13]:
benchmark_rows = []
for cutoff_k in [10, 50, 100, 250]:
    recs_at_k = {
        uid: get_popularity_rec(cutoff_k, seen_movies[uid], popular_movie_ids)
        for uid in sorted(eligible_users)
    }
    assert all(len(items) == cutoff_k for items in recs_at_k.values())
    assert all(len(items) == len(set(items)) for items in recs_at_k.values())
    assert all(set(items).isdisjoint(seen_movies[uid]) for uid, items in recs_at_k.items())
    _, result = evaluate_recommendations(
        recs_at_k, ground_truth, sorted(eligible_users), catalog, k=cutoff_k
    )
    values = result.iloc[:, 0]
    benchmark_rows.append({
        'model': 'MostPopular', 'k': cutoff_k,
        'precision': float(values[f'precision_at_{cutoff_k}']),
        'recall': float(values[f'recall_at_{cutoff_k}']),
        'hitrate': float(values[f'hitrate_at_{cutoff_k}']),
        'ndcg': float(values[f'ndcg_at_{cutoff_k}']),
        'catalog_coverage': float(values[f'catalog_coverage_at_{cutoff_k}']),
        'unique_recommended_items': int(values['unique_recommended_items']),
    })

benchmark_table = pd.DataFrame(benchmark_rows)
output = ROOT / "reports/baselines"
output.mkdir(parents=True, exist_ok=True)
benchmark_table.to_csv(output / "mostpopular_validation.csv", index=False)
display(benchmark_table)

,model,k,precision,recall,hitrate,ndcg,catalog_coverage,unique_recommended_items
0,MostPopular,10,0.091304,0.049683,0.359684,0.108800,0.036705,135
1,MostPopular,50,0.063083,0.171381,0.604743,0.128559,0.106308,391
2,MostPopular,100,0.051403,0.274495,0.717391,0.154932,0.167482,616
3,MostPopular,250,0.036071,0.438946,0.808300,0.202682,0.304513,1120
